In [5]:
import GtoTmodel as GtoTmodel
import Circuits as Circuits 
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [6]:
circuits = Circuits.Circuits()


Graph 4['VDD', 'VSS', 'VIN1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B']
Graph 6['VDD', 'VSS', 'VOUT1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'R1', 'R1_P', 'R1_N', 'C1', 'C1_P', 'C1_N']
Graph 9['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B']
Graph 14['VDD', 'VSS', 'VIN1', 'VOUT1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'R1', 'R1_P', 'R1_N']
Graph 17['VDD', 'VSS', 'VIN1', 'VOUT1', 'R1', 'R1_P', 'R1_N', 'R2', 'R2_P', 'R2_N', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'NM2', 'NM2_D', 'NM2_G', 'NM2_S', 'NM2_B']
Graph 20['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_S', 'NM1_B', 'NM2', 'NM2_D', 'NM2_G', 'NM2_S', 'NM2_B', 'R1', 'R1_P', 'R1_N']
Graph 22['VDD', 'VSS', 'VIN1', 'VOUT1', 'VB1', 'PM1', 'PM1_D', 'PM1_G', 'PM1_S', 'PM1_B', 'PM2', 'PM2_D', 'PM2_G', 'PM2_S', 'PM2_B']
Graph 24['VDD', 'VSS', 'VIN1', 'VOUT1', 'NM1', 'NM1_D', 'NM1_G', 'NM1_

In [7]:
# Assuming circuits has a method or attribute to get the matrix, e.g., circuits.get_matrix()
matrix = circuits.component_lists  # Replace with the actual method or attribute
max_length = max(len(vector) for vector in matrix)
print("Maximum length of vectors in the matrix:", max_length)

Maximum length of vectors in the matrix: 310


In [8]:
circuits.vocab.__len__()  # This should give the number of components

892

In [9]:
torch.manual_seed(1337)
torch.cuda.manual_seed(1337)
embed_dim = 16  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers = 2  # Number of transformer layers
dropout = 0.1  # Dropout rate

#graph_colomns=5
num_components=5
batch_size = 16  # Batch size

graph_input_dim = 310  # Number of colomns in the graph
text_vocab_size = 894  # Vocabulary size for text


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Data Load

In [11]:
num_ciruits = 320
graph_dataset =  circuits.graphs
text_dataset = circuits.component_indices

# Convert graph_dataset and text_dataset to numpy arrays
graph_dataset = np.array(graph_dataset, dtype=object)
text_dataset = np.array(text_dataset, dtype=object)

# Convert numpy arrays to PyTorch tensors
graph_dataset = [torch.tensor(graph, dtype=torch.float32) for graph in graph_dataset]
graph_dataset = [torch.cat((torch.nn.functional.pad(graph, (0, max_length - graph.size(0))),torch.tensor([[9]*310]))) for graph in graph_dataset]
text_dataset = [torch.tensor(text+[893], dtype=torch.int) for text in text_dataset]



In [13]:
graph_dataset[1]

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 1.,  ..., 0., 0., 0.],
        [0., 1., 0.,  ..., 0., 0., 0.],
        [9., 9., 9.,  ..., 9., 9., 9.]])

In [26]:
# Combine graph_dataset and text_dataset into a single list of tuples
combined_dataset = list(zip(graph_dataset, text_dataset))

# Shuffle the combined dataset
np.random.shuffle(combined_dataset)

# Unzip the shuffled dataset back into graph_dataset and text_dataset
graph_dataset, text_dataset = zip(*combined_dataset)

# Convert back to the original data types
graph_dataset = list(graph_dataset)
text_dataset = list(text_dataset)

In [27]:
graph_dataset = torch.cat(graph_dataset).to(device)
text_dataset = torch.cat(text_dataset,).to(device)

In [28]:
graph_train = graph_dataset[:int(0.8 * len(graph_dataset))]
graph_val = graph_dataset[int(0.8 * len(graph_dataset)):]
text_train = text_dataset[:int(0.8 * len(text_dataset))]
text_val = text_dataset[int(0.8 * len(text_dataset)):]

### Generate Batchers

In [29]:
batch_size = 16
block_size = 16


def get_batch( batch_size=4, block_size=16, train=True):

    if train:
        graph_dataset = graph_train
        text_dataset = text_train
    else:
        graph_dataset = graph_val
        text_dataset = text_val
    
    start_indices = torch.randint(0, len(graph_dataset) - block_size, (batch_size,))
    """
    Get a batch of sequences from the text_dataset.

    Args:
        text_dataset (torch.Tensor): The dataset containing text sequences.
        batch_size (int): The number of sequences in the batch.
        block_size (int): The length of each sequence block.

    Returns:
        torch.Tensor: A batch of sequences with shape (batch_size, block_size).
    """
    # Combine graph data and text data for the batch
    
    batch_graph = torch.stack([graph_dataset[i:i + block_size] for i in start_indices])
    batch_text = torch.stack([text_dataset[i:i + block_size] for i in start_indices])
    # Extract sequences of length block_size starting from the sampled indices

    return batch_text, batch_graph 




### Importing the model

In [30]:
model = GtoTmodel.GraphToTextTransformer(
    graph_input_dim, 
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

model = model.to(device)
batch_text, batch_graph = get_batch( batch_size=4, block_size=16)
batch_text = batch_text.to(device)
batch_graph = batch_graph.to(device)


/home/nithira/circuits_gen/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [31]:
learning_rate = 0.001
num_epochs = 10

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

### Training

In [32]:
# model.to('cpu')
# atch_text = batch_text.to('cpu')
# batch_graph = batch_graph.to('cpu')
batch_text, batch_graph = get_batch( batch_size=4, block_size=16)
for _ in range(100):
    batch_text, batch_graph = get_batch(batch_size=4, block_size=16)

    # Forward pass through the model
    # Ensure batch_text has at least two dimensions
    if batch_text.dim() == 1:
        batch_text = batch_text.unsqueeze(0)  # Add a batch dimension if missing

    # Ensure batch_graph has at least two dimensions
    if batch_graph.dim() == 1:
        batch_graph = batch_graph.unsqueeze(0)  # Add a batch dimension if missing

    # Forward pass through the model
    output = model(batch_graph, batch_text[:, :-1])  # Exclude the last token for input

    print("Output shape:", output.shape)  # Should be (batch_size, seq_length, vocab_size)

    # Calculate loss
    loss = criterion(output.reshape(-1, text_vocab_size), batch_text[:, 1:].reshape(-1).long())  # Exclude the first token for target
    print("Loss:", loss.item())  # Print the loss value
    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    # Print the updated model parameter

Output shape: torch.Size([4, 15, 894])
Loss: 7.088229179382324
Output shape: torch.Size([4, 15, 894])
Loss: 6.963825702667236
Output shape: torch.Size([4, 15, 894])
Loss: 6.857078552246094
Output shape: torch.Size([4, 15, 894])
Loss: 6.788712024688721
Output shape: torch.Size([4, 15, 894])
Loss: 6.762427806854248
Output shape: torch.Size([4, 15, 894])
Loss: 6.755639553070068
Output shape: torch.Size([4, 15, 894])
Loss: 6.8518500328063965
Output shape: torch.Size([4, 15, 894])
Loss: 6.913756370544434
Output shape: torch.Size([4, 15, 894])
Loss: 6.6688079833984375
Output shape: torch.Size([4, 15, 894])
Loss: 6.762810230255127
Output shape: torch.Size([4, 15, 894])
Loss: 6.67926025390625
Output shape: torch.Size([4, 15, 894])
Loss: 6.663780212402344
Output shape: torch.Size([4, 15, 894])
Loss: 6.783738613128662
Output shape: torch.Size([4, 15, 894])
Loss: 6.7389750480651855
Output shape: torch.Size([4, 15, 894])
Loss: 6.599760055541992
Output shape: torch.Size([4, 15, 894])
Loss: 6.762572

### Saving the Model

In [33]:
# Define the file path to save the model and hyperparameters
save_path = "model_checkpoint.pth"

# Create a dictionary to store the model state and hyperparameters
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'embed_dim': embed_dim,
    'num_heads': num_heads,
    'num_layers': num_layers,
    'dropout': dropout,
    'learning_rate': learning_rate,
    'text_vocab_size': text_vocab_size,
    'graph_input_dim': graph_input_dim,
    'batch_size': batch_size,
    'block_size': block_size
}

# Save the checkpoint
torch.save(checkpoint, save_path)
print(f"Model and hyperparameters saved to {save_path}")

Model and hyperparameters saved to model_checkpoint.pth


### Load the Model

In [34]:
# Define the file path to load the model and hyperparameters
load_path = "model_checkpoint.pth"

# Load the checkpoint
checkpoint = torch.load(load_path)

# Restore the model state and hyperparameters
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Restore hyperparameters if needed
embed_dim = checkpoint['embed_dim']
num_heads = checkpoint['num_heads']
num_layers = checkpoint['num_layers']
dropout = checkpoint['dropout']
learning_rate = checkpoint['learning_rate']
text_vocab_size = checkpoint['text_vocab_size']
graph_input_dim = checkpoint['graph_input_dim']
batch_size = checkpoint['batch_size']
block_size = checkpoint['block_size']

print(f"Model and optimizer state loaded from {load_path}")

Model and optimizer state loaded from model_checkpoint.pth


### Validation loop

In [ ]:
model.eval()  # Set the model to evaluation mode
total_val_loss = 0

with torch.no_grad():  # Disable gradient computation for validation
    for _ in range(1000):  # Number of validation iterations
        batch_text, batch_graph = get_batch(batch_size=4, block_size=16, train=False)

        # Ensure batch_text has at least two dimensions
        if batch_text.dim() == 1:
            batch_text = batch_text.unsqueeze(0)  # Add a batch dimension if missing

        # Ensure batch_graph has at least two dimensions
        if batch_graph.dim() == 1:
            batch_graph = batch_graph.unsqueeze(0)  # Add a batch dimension if missing

        # Forward pass through the model
        output = model(batch_graph, batch_text[:, :-1])  # Exclude the last token for input

        # Calculate loss
        loss = criterion(output.reshape(-1, text_vocab_size), batch_text[:, 1:].reshape(-1).long())  # Exclude the first token for target
        total_val_loss += loss.item()

        print("Validation Loss:", loss.item())

# Calculate average validation loss
average_val_loss = total_val_loss / 1000
# Print the average validation loss for the current epoch
print(f"Epoch {_ + 1}, Average Validation Loss: {average_val_loss}")

In [ ]:
model.eval()

def test_model(test_graph, test_text):
    with torch.no_grad(): 

        if test_text.dim() == 1:
            test_text = test_text.unsqueeze(0)

        if test_graph.dim() == 1:
            test_graph = test_graph.unsqueeze(0)

        # Forward pass through the model
        output = model(test_graph, test_text[:, :-1])  # Exclude the last token for input

        predicted_tokens = torch.argmax(output, dim=-1)

        return predicted_tokens

# Example: Test the model with a single sample
# Generate random test data
test_graph = torch.rand((1, graph_input_dim), device=device)  # Random graph input
test_text = torch.randint(0, text_vocab_size, (1, 16), device=device)  # Random text input

# test_graph = graph_val[0].unsqueeze(0)
# test_text = text_val[:16].unsqueeze(0) 

predicted_tokens = test_model(test_graph, test_text)
print("Predicted Tokens:", predicted_tokens)
print("Actual Tokens:", test_text[:, 1:])  # Exclude the first token for comparison

RuntimeError: permute(sparse_coo): number of dimensions in the tensor input does not match the length of the desired ordering of dimensions i.e. input.dim() = 2 is not equal to len(dims) = 3